# Method-preserving reference reruns: UIEB and LSUI
Scientific question: how do the five paper reference methods perform on fixed UIEB/LSUI splits under the same external budget and evaluator while retaining their defining inputs, losses, optimizers, and training mechanics? This is not an exact publication reproduction or a unified-loss architecture benchmark.

## 1. Purpose and scientific question

Standardize data, split, seeds, 100 epochs, 256 crops, effective batch 4, validation-PSNR selection, held-out tests, metrics, and reporting. Preserve each method internally.

## 2. Repository/environment inspection

In [ ]:
from pathlib import Path
import json, sys, torch
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks': REPO_ROOT = REPO_ROOT.parent
SRC = REPO_ROOT / 'src'
if str(SRC) not in sys.path: sys.path.insert(0, str(SRC))
DATA_ROOT = REPO_ROOT / 'datasets'
OUTPUT_ROOT = REPO_ROOT / 'outputs' / 'reference_methods'
print(REPO_ROOT, torch.__version__, torch.cuda.is_available())

## 3. Benchmark configuration

In [ ]:
from uwir.training.runner import BenchmarkConfig, MODEL_SEEDS, write_benchmark_metadata
CONFIG = BenchmarkConfig(epochs=100, crop_size=256, effective_batch_size=4, cache_data='auto')
SMOKE = True
RUN_FULL = False
RUN_CROSS_DATASET = False
write_benchmark_metadata(OUTPUT_ROOT, CONFIG, REPO_ROOT)
CONFIG

## 4. Reference method registry

In [ ]:
from uwir.reference_methods import REFERENCE_METHODS
list(REFERENCE_METHODS)

## 5. Reference method provenance

In [ ]:
provenance = {name: cls.provenance() for name, cls in REFERENCE_METHODS.items()}
print(json.dumps(provenance, indent=2))

## 6. Method training configuration report

In [ ]:
training_configs = {name: cls.training_config() for name, cls in REFERENCE_METHODS.items()}
for name, cfg in training_configs.items(): print(name, cfg['input_formulation'], cfg['losses'], cfg['optimizers'], cfg['schedulers'], sep=' | ')

## 7. UIEB discovery/integrity

In [ ]:
from uwir.datasets.uieb import discover_uieb, build_uieb_datasets
uieb_pairs = discover_uieb(DATA_ROOT / 'UIEB')
assert len(uieb_pairs) == 890
print('UIEB valid pairs:', len(uieb_pairs))

## 8. LSUI discovery/integrity

In [ ]:
from uwir.datasets.lsui import compact_tree, discover_lsui, build_lsui_datasets
print(compact_tree(DATA_ROOT / 'LSUI'))
lsui_pairs, lsui_report = discover_lsui(DATA_ROOT / 'LSUI')
print(json.dumps(lsui_report, indent=2))

## 9. Fixed split generation/loading

In [ ]:
uieb = build_uieb_datasets(DATA_ROOT/'UIEB', OUTPUT_ROOT/'splits/uieb_split_manifest.json', cache=CONFIG.cache_data)
lsui = build_lsui_datasets(DATA_ROOT/'LSUI', OUTPUT_ROOT/'splits/lsui_split_manifest.json', cache=CONFIG.cache_data)
DATASETS = {'UIEB': uieb, 'LSUI': lsui}
{k: {s: len(v) for s,v in value.items()} for k,value in DATASETS.items()}

## 10. Split invariance checks

In [ ]:
for name, splits in DATASETS.items():
    ids = [{r.identity for r in splits[key].records} for key in ('train','val','test')]
    assert not (ids[0]&ids[1] or ids[0]&ids[2] or ids[1]&ids[2])
    for seed in MODEL_SEEDS: assert ids == [{r.identity for r in splits[key].records} for key in ('train','val','test')]
print('PASS: split membership is disjoint and model-seed invariant')

## 11. Common paired dataset interface

In [ ]:
sample = uieb['train'][0]
{k: (tuple(v.shape) if hasattr(v,'shape') else v) for k,v in sample.items()} 

## 12. Adapter contract checks

In [ ]:
# Construction checks may download fixed VGG auxiliary weights into OUTPUT_ROOT/cache/torch.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
for name, cls in REFERENCE_METHODS.items():
    adapter = cls(device, amp=True)
    output = adapter.inference(torch.rand(1,3,32,32,device=device))
    assert output.shape == (1,3,32,32) and torch.isfinite(output).all() and output.min() >= 0 and output.max() <= 1
print('PASS')

## 13–17. Method configuration inspection

In [ ]:
for method in ('funie_gan','ucolor','unet','water_net','uwformer'):
    print('\n', method.upper()); print(json.dumps(provenance[method], indent=2)); print(json.dumps(training_configs[method], indent=2))

## 18. Common evaluator sanity checks

In [ ]:
from uwir.evaluation.quality_metrics import evaluate_pair
x = torch.full((3,32,32), 0.5)
metric_check = evaluate_pair(x, x)
assert metric_check['ssim'] == 1 and metric_check['ciede2000'] == 0
metric_check

## 19. Full 10-combination smoke test

In [ ]:
from uwir.training.runner import run_reference_experiment
if SMOKE:
    for dataset_name, datasets in DATASETS.items():
        for method in REFERENCE_METHODS:
            microbatches = 4 if method in {'ucolor','uwformer'} else 1
            run_reference_experiment(dataset_name=dataset_name, method_name=method, datasets=datasets, output_root=OUTPUT_ROOT, repository_root=REPO_ROOT, model_seed=0, config=CONFIG, device=device, smoke=True, max_train_batches=microbatches, max_eval_samples=1, resume=True)
print('Smoke matrix complete')

## 20. Preflight PASS/FAIL report

In [ ]:
checks = {'UIEB 890/720/80/90': len(uieb_pairs)==890 and [len(uieb[x]) for x in ('train','val','test')]==[720,80,90], 'LSUI pairing': lsui_report['matched_pairs']==len(lsui_pairs), 'split manifests frozen': all((OUTPUT_ROOT/'splits'/x).exists() for x in ('uieb_split_manifest.json','lsui_split_manifest.json')), 'external budget': (CONFIG.epochs,CONFIG.crop_size,CONFIG.effective_batch_size)==(100,256,4), 'selection': CONFIG.model_selection_metric=='val_psnr'}
for key,value in checks.items(): print('PASS' if value else 'FAIL', key)
assert all(checks.values())

## 21–22. Full benchmark launch/resume and held-out evaluation

In [ ]:
if RUN_FULL:
    for dataset_name, datasets in DATASETS.items():
        for method in REFERENCE_METHODS:
            for seed in MODEL_SEEDS:
                run_reference_experiment(dataset_name=dataset_name, method_name=method, datasets=datasets, output_root=OUTPUT_ROOT, repository_root=REPO_ROOT, model_seed=seed, config=CONFIG, device=device, resume=True)
print('RUN_FULL is', RUN_FULL)

## 23. Efficiency benchmark

In [ ]:
from uwir.evaluation.inference_benchmark import benchmark_inference
# Run once per method after smoke: efficiency = [benchmark_inference(cls(device)) for cls in REFERENCE_METHODS.values()]


## 24. Aggregate results

In [ ]:
from uwir.training.runner import aggregate_results
if (OUTPUT_ROOT/'per_run_results.csv').exists(): aggregate_results(OUTPUT_ROOT/'per_run_results.csv', OUTPUT_ROOT/'aggregate_results.csv')

## 25. Optional cross-dataset evaluation

In [ ]:
from uwir.training.runner import evaluate_cross_dataset
if RUN_CROSS_DATASET:
    for method in REFERENCE_METHODS:
        for seed in MODEL_SEEDS:
            evaluate_cross_dataset(source_dataset='UIEB', target_dataset='LSUI', method_name=method, model_seed=seed, target_test_dataset=lsui['test'], output_root=OUTPUT_ROOT, device=device)
            evaluate_cross_dataset(source_dataset='LSUI', target_dataset='UIEB', method_name=method, model_seed=seed, target_test_dataset=uieb['test'], output_root=OUTPUT_ROOT, device=device)
print('RUN_CROSS_DATASET is', RUN_CROSS_DATASET)

## 26. Final comparison table

In [ ]:
import pandas as pd
path = OUTPUT_ROOT/'aggregate_results.csv'
if path.exists():
    table = pd.read_csv(path)
    display(table[['dataset','method','psnr_mean','psnr_std','ssim_mean','ssim_std','ciede2000_mean','ciede2000_std','uciqe_mean','uciqe_std','uiqm_mean','uiqm_std']])

## 27. Reproducibility report

In [ ]:
for filename in ('benchmark_config.json','environment.json','method_provenance.json','method_training_configs.json'):
    print(filename, json.loads((OUTPUT_ROOT/filename).read_text()))